### Get Real Objective Metrics From Filtered Transcript

In [79]:
def parse_transcript(transcipt_name):

    chat_stats = {}
    policy = None
    dialogs = {}
    DIALOG_TYPES = ['cts', 'hdc', 'faq']

    user_dialog_nums = {}

    with open(transcipt_name, "r") as transcript:
        for line in transcript:
            if "(POLICY:" in line:
                current_dialog = {}
                tmp = line.split()
                user = tmp[1].strip()
                policy = tmp[3].strip(")").strip()
                current_dialog['user'] = user
                current_dialog['turns'] = []
                # TODO: add back goals to dialogs and analyze if there is anything interesting at a per/goal level
                # goal_type = tmp[-1].strip(")").strip()
            elif "USER:" in line and not "POST-NLU" in line:
                current_dialog["turns"].append(line)
            elif "SYSTEM" in line:
                current_dialog['turns'].append(line)
            elif "DIALOG END:" in line:
                current_dialog["end_condition"] = line.split(":")[1].strip()
            elif "SUBJECTIVE LENGTH" in line:
                current_dialog["sub_length"] = line.split(":")[1].strip()
            elif "SUBJECTIVE QUALITY" in line:
                current_dialog["sub_quality"] = line.split(":")[1].strip()
            elif line.strip() == "":
                # TODO: Change this line to analyse one group at a time
                if current_dialog and policy in DIALOG_TYPES:
                    obj_length = len(current_dialog["turns"])
                    if not policy in chat_stats:
                        chat_stats[policy] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}
                    chat_stats[policy]["length"].append(obj_length)
                    chat_stats[policy]["end_condition"].append(current_dialog["end_condition"])
                    chat_stats[policy]["sub_length"].append(int(current_dialog["sub_length"]))
                    chat_stats[policy]["sub_quality"].append(int(current_dialog["sub_quality"]))
                    current_dialog["length"] = obj_length
                    if current_dialog['user'] not in user_dialog_nums:
                        user_dialog_nums[current_dialog['user']] = 0
                    user_dialog_nums[current_dialog['user']] += 1
                    if policy not in dialogs:
                        dialogs[policy] = []
                    dialogs[policy].append(current_dialog)
                    current_dialog = {}
    return (chat_stats, dialogs, user_dialog_nums)


In [80]:
ling_ad_chat_stats, ling_ad_dialogs, ling_ad_user_dialog_nums = parse_transcript("combined/combined_transcript.txt")
baseline_chat_stats, baseline_dialogs, baseline_user_dialog_nums = parse_transcript("../mental_models/combined_data/transcript.txt")

In [81]:
import csv
success_counts = {"baseline": {}, "ling_ad": {}}
with open("chat_stats.tsv", "w") as outfile:
    writer = csv.writer(outfile, delimiter='\t',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["dialog_system", "ling_style", "length", "success", "sub_length", "sub_success"])
    for policy in ling_ad_chat_stats:
        success_counts["ling_ad"][policy] = 0
        stats = ling_ad_chat_stats[policy]
        for i in ling_ad_chat_stats[policy]["length"]:
            length = stats["length"][i]
            success = 1 if "SUCCESS" in stats["end_condition"][i] else 0
            print(stats["end_condition"][i])
            success_counts["ling_ad"][policy] += success
            sub_len = stats["sub_length"][i]
            sub_success = stats["sub_quality"][i]
            writer.writerow([policy, "ling_ad", length, success, sub_len, sub_success])
    for policy in baseline_chat_stats:
        success_counts["baseline"][policy] = 0
        stats = baseline_chat_stats[policy]
        for i in baseline_chat_stats[policy]["length"]:
            length = stats["length"][i]
            success = 1 if "SUCCESS" in stats["end_condition"][i] else 0
            success_counts["baseline"][policy] += 1
            sub_len = stats["sub_length"][i]
            sub_success = stats["sub_quality"][i]
            writer.writerow([policy, "baseline", length, success, sub_len, sub_success])

### Collect corpus statistics per/policy
* Number 
* Length
* #Success
* Avg. first utterance length
* Avg. all user utterance lengths

In [88]:
def calc_dialog_metrics(chat_stats, dialogs):
    num_dialogs = {}
    success = {}
    avg_len_dialog = {}
    avg_len_first_utterance = {}
    avg_len_all_utterances = {}
    avg_sub_quality = {}
    avg_sub_len = {}

    for policy in chat_stats:
        num_dialogs[policy] = 0
        success[policy] = 0
        avg_len_dialog[policy] = 0
        avg_len_first_utterance[policy] = 0
        avg_len_all_utterances[policy] = 0
        avg_sub_quality[policy] = 0
        avg_sub_len[policy] = 0
        
        success[policy] = chat_stats[policy]["end_condition"].count("SUCCESS") + chat_stats[policy]["end_condition"].count("SUCCESS - OTHER QUESTION")
        avg_len_dialog[policy] = sum(chat_stats[policy]["length"])
        avg_sub_quality[policy] = sum(chat_stats[policy]["sub_quality"])
        avg_sub_len[policy] = sum(chat_stats[policy]["sub_length"])

        for record in dialogs[policy]:
            num_dialogs[policy] += 1
            d = record['turns']
            user_turns = [t for t in d if "USER" in t]
            len_user_turns = [len(t[6:].split()) for t in user_turns]
            avg_len_first_utterance[policy] += len_user_turns[0]
            avg_len_all_utterances[policy] += sum(len_user_turns)/len(len_user_turns)

        avg_len_dialog[policy] = avg_len_dialog[policy]/num_dialogs[policy]
        avg_sub_len[policy] = avg_sub_len[policy]/num_dialogs[policy]
        avg_sub_quality[policy] = avg_sub_quality[policy]/num_dialogs[policy]

        avg_len_first_utterance[policy] = avg_len_first_utterance[policy]/num_dialogs[policy]
        avg_len_all_utterances[policy] = avg_len_all_utterances[policy]/num_dialogs[policy]

        print(policy)
        print(f"NUM DIALOGS: {num_dialogs[policy]}")
        print(f"COUNT SUCCESS: {success[policy]}")
        print(f"PERCENT SUCCESS: {success[policy]/num_dialogs[policy]*100}")
        print(f"SUBJECTIVE QUALITY: {avg_sub_quality[policy]}")
        print(f"AVG NUM TURNS: {avg_len_dialog[policy]}")
        print(f"SUBJECTIVE LENGTH: {avg_sub_len[policy]}")

        print(f"AVG LEN INITIAL UTTERANCE: {avg_len_first_utterance[policy]}")
        print(f"AVG LEN ALL UTTERANCES: {avg_len_all_utterances[policy]}")

In [89]:
print("BASELINE STATS:")
calc_dialog_metrics(baseline_chat_stats, baseline_dialogs)

BASELINE STATS:
cts
NUM DIALOGS: 61
COUNT SUCCESS: 47
PERCENT SUCCESS: 77.04918032786885
SUBJECTIVE QUALITY: 2.8688524590163933
AVG NUM TURNS: 7.377049180327869
SUBJECTIVE LENGTH: 2.918032786885246
AVG LEN INITIAL UTTERANCE: 8.721311475409836
AVG LEN ALL UTTERANCES: 6.361865729898516
hdc
NUM DIALOGS: 66
COUNT SUCCESS: 29
PERCENT SUCCESS: 43.93939393939394
SUBJECTIVE QUALITY: 2.409090909090909
AVG NUM TURNS: 13.318181818181818
SUBJECTIVE LENGTH: 3.0757575757575757
AVG LEN INITIAL UTTERANCE: 8.227272727272727
AVG LEN ALL UTTERANCES: 5.370540780768053
faq
NUM DIALOGS: 61
COUNT SUCCESS: 35
PERCENT SUCCESS: 57.377049180327866
SUBJECTIVE QUALITY: 2.6065573770491803
AVG NUM TURNS: 2.262295081967213
SUBJECTIVE LENGTH: 2.278688524590164
AVG LEN INITIAL UTTERANCE: 10.163934426229508
AVG LEN ALL UTTERANCES: 10.248633879781423


In [90]:
print("LINGUISTIC ADAPTATION STATS:")
calc_dialog_metrics(ling_ad_chat_stats, ling_ad_dialogs)

LINGUISTIC ADAPTATION STATS:
cts
NUM DIALOGS: 64
COUNT SUCCESS: 57
PERCENT SUCCESS: 89.0625
SUBJECTIVE QUALITY: 3.15625
AVG NUM TURNS: 6.4375
SUBJECTIVE LENGTH: 3.078125
AVG LEN INITIAL UTTERANCE: 11.09375
AVG LEN ALL UTTERANCES: 8.035714285714286
faq
NUM DIALOGS: 65
COUNT SUCCESS: 41
PERCENT SUCCESS: 63.07692307692307
SUBJECTIVE QUALITY: 2.7384615384615385
AVG NUM TURNS: 2.6769230769230767
SUBJECTIVE LENGTH: 2.646153846153846
AVG LEN INITIAL UTTERANCE: 11.723076923076922
AVG LEN ALL UTTERANCES: 11.624358974358975
hdc
NUM DIALOGS: 62
COUNT SUCCESS: 34
PERCENT SUCCESS: 54.83870967741935
SUBJECTIVE QUALITY: 2.629032258064516
AVG NUM TURNS: 12.53225806451613
SUBJECTIVE LENGTH: 2.725806451612903
AVG LEN INITIAL UTTERANCE: 7.548387096774194
AVG LEN ALL UTTERANCES: 5.356913606107155


### Test Whether linguistic templates had an effect on any of these metrics

In [94]:
from scipy.stats import mannwhitneyu
from scipy.stats import barnard_exact
from scipy.stats import ttest_ind

def run_dialog_stats(condition: str, baseline_chat_stats, baseline_dialogs, ling_ad_chat_stats, ling_ad_dialogs):
    # For the Subjective questions, we have unpaired ordinal data, so we will use a Mann Whitney U test to compare them
    base_sub_len = baseline_chat_stats[condition]["sub_length"]
    base_sub_quality = baseline_chat_stats[condition]["sub_quality"]
    exp_sub_len = ling_ad_chat_stats[condition]["sub_length"]
    exp_sub_quality = ling_ad_chat_stats[condition]["sub_quality"]

    print("Subjective Length")
    print(mannwhitneyu(base_sub_len, exp_sub_len))

    print("Subjective Quality")
    print(mannwhitneyu(base_sub_quality, exp_sub_quality, alternative='less'))

    # For Dialog length, we can use a T-test
    base_length = baseline_chat_stats[condition]["length"]
    exp_length = ling_ad_chat_stats[condition]["length"]
    print("Dialog Length")
    print(ttest_ind(base_length, exp_length))

    # For Dialog Success, we can use a Fisher Exact Test or more powerful Barnard Exact Test
    table = []
    base_success = sum([1 for el in baseline_chat_stats[condition]["end_condition"] if "SUCCESS" in el])
    base_faliure = sum([1 for el in baseline_chat_stats[condition]["end_condition"] if "FAILURE" in el])
    exp_success = sum([1 for el in ling_ad_chat_stats[condition]["end_condition"] if "SUCCESS" in el])
    exp_failure = sum([1 for el in ling_ad_chat_stats[condition]["end_condition"] if "FAILURE" in el])
    table.append([base_success, exp_success])
    table.append([base_faliure, exp_failure])
    print(table)
    print("Objective Success")
    # print(table)
    
    print(barnard_exact(table=table, alternative="less"))


print("CTS")
run_dialog_stats("cts", baseline_chat_stats=baseline_chat_stats, baseline_dialogs=baseline_dialogs, ling_ad_chat_stats=ling_ad_chat_stats, ling_ad_dialogs=ling_ad_dialogs)

print("\nFAQ")
run_dialog_stats("faq", baseline_chat_stats=baseline_chat_stats, baseline_dialogs=baseline_dialogs, ling_ad_chat_stats=ling_ad_chat_stats, ling_ad_dialogs=ling_ad_dialogs)

print("\nHDC")
run_dialog_stats("hdc", baseline_chat_stats=baseline_chat_stats, baseline_dialogs=baseline_dialogs, ling_ad_chat_stats=ling_ad_chat_stats, ling_ad_dialogs=ling_ad_dialogs)



CTS
Subjective Length
MannwhitneyuResult(statistic=1736.0, pvalue=0.19666782719888964)
Subjective Quality
MannwhitneyuResult(statistic=1613.5, pvalue=0.03874164307376467)
Dialog Length
Ttest_indResult(statistic=0.7159318239074978, pvalue=0.7623047853406988)
[[47, 57], [14, 7]]
Objective Success
BarnardExactResult(statistic=-1.7957531119041419, pvalue=0.04315725024685003)

FAQ
Subjective Length
MannwhitneyuResult(statistic=1461.0, pvalue=0.004921242735745279)
Subjective Quality
MannwhitneyuResult(statistic=1803.5, pvalue=0.18215068880735946)
Dialog Length
Ttest_indResult(statistic=-1.6000725473382151, pvalue=0.05606364886265712)
[[35, 41], [26, 24]]
Objective Success
BarnardExactResult(statistic=-0.6535522551480227, pvalue=0.2834765691434432)

HDC
Subjective Length
MannwhitneyuResult(statistic=2402.0, pvalue=0.07540631838397686)
Subjective Quality
MannwhitneyuResult(statistic=1818.0, pvalue=0.13036637453497152)
Dialog Length
Ttest_indResult(statistic=0.440474662841951, pvalue=0.66982594

### Summary of findings:

- In the CTS condition, there was a significant improvement in the subjective performance
- In FAQ, there was a significant improvement in perceived appropriateness of the dialog length

### Check if subjective quality changed between only successful dialogs

In [69]:
base_pos_sub_succ = {}
ling_ad_pos_sub_succ = {}
for policy in baseline_dialogs:
    for dialog in baseline_dialogs[policy]:
        if "SUCCESS" in dialog["end_condition"]:
            if not policy in base_pos_sub_succ:
                base_pos_sub_succ[policy] = []
            base_pos_sub_succ[policy].append(int(dialog["sub_quality"]))
    for dialog in ling_ad_dialogs[policy]:
        if "SUCCESS" in dialog["end_condition"]:
            if not policy in ling_ad_pos_sub_succ:
                ling_ad_pos_sub_succ[policy] = []
            ling_ad_pos_sub_succ[policy].append(int(dialog["sub_quality"]))

for policy in base_pos_sub_succ:
    print(policy)
    print(mannwhitneyu(base_pos_sub_succ[policy], ling_ad_pos_sub_succ[policy]))

cts
MannwhitneyuResult(statistic=1222.0, pvalue=0.4112511375889646)
hdc
MannwhitneyuResult(statistic=406.0, pvalue=0.27299342784037683)
faq
MannwhitneyuResult(statistic=715.0, pvalue=0.9825799692696601)


**Facit**
No, there is not difference in perceived quality when we only look at successful dialogs

### Parse Surveys

In [13]:
import ast
import csv

def parse_surveys(survey_file, outfile, user_dialog_nums):
    post_surveys = []
    pre_surveys = []
    users = set()
    with open(survey_file, "r") as infile:
        for line in infile:
            if "PREFERRED_STYLE" not in line:
                user, survey = line.split("||")
                user = user.split(":")[1].strip()
                users.add(user)
                survey = survey[13:].strip()
                survey = ast.literal_eval(survey)
                survey["user"] = user
                if "POST-SURVEY" in line:
                    post_surveys.append(survey)
                else:
                    pre_surveys.append(survey)   

    # Remove Users who don't interact with the system
    user_black_list = set()
    user_gray_list = set()

    for user in users:
        if user in user_dialog_nums:
            if user_dialog_nums[user] != 3:
                user_gray_list.add(user)
        else:
            user_black_list.add(user)

    print(f"Removed {len(user_black_list)} users: {user_black_list}")
    print("To Investigate: ", user_gray_list)
    users = [user for user in users if user not in user_black_list]
    print(len(users))

    if not outfile is None:
        # Save surveys to CSV for easier conent analysis
        with open("pre_survey_ling_ad.csv", "w", newline='') as outfile:
            fieldnames = pre_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in pre_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

        with open("post_survey_ling_ad.csv", "w", newline='') as outfile:
            fieldnames = post_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in post_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

    return pre_surveys, post_surveys, user_black_list

In [14]:
baseline_pre_surveys, baseline_post_surveys, base_blacklist = parse_surveys("../mental_models/combined_data/survey_log_full.txt", None, baseline_user_dialog_nums)
exp_pre_surveys, exp_post_surveys, exp_blacklist = parse_surveys("combined/combined_survey_log.txt", None, ling_ad_user_dialog_nums)

Removed 2 users: {'6e6ed1b576c8f271b519a059d5bf0d', '1fd4736bcd456d02458c2a32944e60'}
To Investigate:  {'000650684db95bdac754d1c314c481', '2e4bd44563cdfd591a0a4bd566c308', '85452be9505ef3ee3499d59b5300f3', 'e5de0b4b0679651ef175fceeb3413f'}
64
Removed 4 users: {'deeaf6cd3b0de1978289795e7a96df', '81e4a5dc87e754bc8f5ca09ea06b5c', 'aa0711803864dcdab860f4e2443f8e', '24413921ec243ff743a5a53ffbb51c'}
To Investigate:  {'680364a9f4d189548e98bc745564ec', 'b284d3f286ce58f9587ec7c8bdc73c', '69e746ae2e242f850a4a29a9e50ad5', '750c51071a48cb39f6b7ab22bef5f6', 'c334bbab21038ec5b0e8ef2532ca98', '7ad16eb7438e5306d32d36c3293364', 'bef4f95c2e4d20aa7762ff58cea744'}
66


### Parse out Trust and Usability scores

In [15]:
import numpy as np

def parse_trust_and_usability_scores(post_surveys, user_black_list):
    trust = []
    reliability = []
    usability = []

    u_usability = {}
    u_trust = {}
    u_reliability = {}

    for res in post_surveys:
        user = res["user"]
        if user in user_black_list or user == "47e68725c26f72d805709141e76fd0" or user == "57affbcf1a53cf8152a4f84b337572":
            continue
        user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
        user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
        # should be 0 to 4 scale, not 1 to 5
        user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
        u_usability[user] = user_usability
        trust.append(user_trust)
        u_trust[user] = user_trust
        reliability.append(user_reliability)
        u_reliability[user] = user_reliability
        usability.append(user_usability)
        
        # print(f" USER: {user}: Trust: {user_trust} Reliability: {user_reliability} Usability: {user_usability}")
        
    print(f"TRUST: {np.mean(trust)} +/- {np.std(trust)}")
    print(f"RELIABILITY: {np.mean(reliability)} +/- {np.std(reliability)}")
    print(f"USABILITY: {np.mean(usability)} +/- {np.std(usability)}")
    return trust, reliability, usability

In [16]:
def create_user_condition_mapping(user_file):
    user_condition_mapping = {}
    with open(user_file, "r") as infile:
        for line in infile:
            if "GROUP" in line:
                user, group = line.split("||")
                user = user.split(":")[1].strip()
                group = group.split(":")[1].strip()
                user_condition_mapping[user] = group
    return user_condition_mapping


user_condition_mapping = create_user_condition_mapping("combined/combined_user_log.txt") | create_user_condition_mapping("../mental_models/combined_data/user_log_full.txt")

In [87]:
from scipy.stats import ttest_ind
print('BASELINE')
baseline_trust, baseline_reliability, baseline_usability = parse_trust_and_usability_scores(baseline_post_surveys, base_blacklist)
print("\nLING AD")
exp_trust, exp_reliability, exp_usability = parse_trust_and_usability_scores(exp_post_surveys, exp_blacklist)

print("trust")
print(ttest_ind(baseline_trust, exp_trust, alternative="less"))

print("reliability")
print(ttest_ind(baseline_reliability, exp_reliability, alternative="less"))

print("usability")
print(ttest_ind(baseline_usability, exp_usability, alternative="less"))

BASELINE
TRUST: 2.8548387096774195 +/- 0.968816607592336
RELIABILITY: 2.71505376344086 +/- 0.7984006732852397
USABILITY: 51.91532258064516 +/- 26.195909845582324

LING AD
TRUST: 2.8923076923076922 +/- 1.2105062758147858
RELIABILITY: 2.864102564102564 +/- 0.8716289434854894
USABILITY: 55.86538461538461 +/- 28.69092850882008
trust
Ttest_indResult(statistic=-0.19050601683153043, pvalue=0.42461083988908405)
reliability
Ttest_indResult(statistic=-0.9955716004188103, pvalue=0.16069083282314392)
usability
Ttest_indResult(statistic=-0.802705748933264, pvalue=0.21183420502283629)


In [2]:
from scipy.stats import tukey_hsd
def scores_per_condition(user_condition_mapping, surveys, blacklist):
    condition_surveys = {}
    for survey in surveys:
        user = survey["user"]
        condition = user_condition_mapping[user]
        if condition not in condition_surveys:
            condition_surveys[condition] = []
        condition_surveys[condition].append(survey)

    assert(len(condition_surveys) == 3)

    for condition in condition_surveys:
        print(condition)
        parse_trust_and_usability_scores(condition_surveys[condition], blacklist)

In [1]:
def stats_per_condition(user_condition_mapping, base_surveys, ling_ad_surveys, base_blacklist, ling_ad_blacklist):
    condition_base_surveys = {}
    condition_ling_ad_surveys = {}
    for survey in base_surveys:
        user = survey["user"]
        condition = user_condition_mapping[user]
        if condition not in condition_base_surveys:
            condition_base_surveys[condition] = []
        condition_base_surveys[condition].append(survey)
    for survey in ling_ad_surveys:
        user = survey["user"]
        condition = user_condition_mapping[user]
        if condition not in condition_ling_ad_surveys:
            condition_ling_ad_surveys[condition] = []
        condition_ling_ad_surveys[condition].append(survey)

    for condition in condition_base_surveys:
        print(condition)
        print("BASELINE")
        base_trust, base_reliability, base_usability = parse_trust_and_usability_scores(condition_base_surveys[condition], base_blacklist)
        print("LING AD")
        trust, reliability, usability = parse_trust_and_usability_scores(condition_ling_ad_surveys[condition], ling_ad_blacklist)
        print("trust", mannwhitneyu(base_trust, trust))
        print("reliability", mannwhitneyu(base_reliability, reliability))
        print("usability", mannwhitneyu(base_usability, usability))

In [60]:
stats_per_condition(user_condition_mapping, baseline_post_surveys, exp_post_surveys, base_blacklist, exp_blacklist)

# print("\nLING AD")
# stats_per_condition(user_condition_mapping, exp_post_surveys, exp_blacklist)

cts
BASELINE
TRUST: 3.1578947368421053 +/- 0.8119604537127112
RELIABILITY: 2.9649122807017547 +/- 0.7083027802872404
USABILITY: 62.828947368421055 +/- 23.073260416776552
LING AD
TRUST: 3.0454545454545454 +/- 1.1957224034514462
RELIABILITY: 3.1212121212121207 +/- 0.8183220979630773
USABILITY: 65.3409090909091 +/- 24.838060220902722
trust MannwhitneyuResult(statistic=215.0, pvalue=0.8811624844973225)
reliability MannwhitneyuResult(statistic=187.5, pvalue=0.5812668010662736)
usability MannwhitneyuResult(statistic=193.5, pvalue=0.6927679422393582)
hdc
BASELINE
TRUST: 2.6136363636363638 +/- 1.1071238144380546
RELIABILITY: 2.424242424242424 +/- 0.8728903467172167
USABILITY: 36.93181818181818 +/- 25.969079277332817
LING AD
TRUST: 2.5476190476190474 +/- 1.100916954447323
RELIABILITY: 2.468253968253968 +/- 0.8476844395440436
USABILITY: 41.964285714285715 +/- 29.00526283467956
trust MannwhitneyuResult(statistic=234.5, pvalue=0.9408944317856965)
reliability MannwhitneyuResult(statistic=216.0, pva

### Summary of Findings:
- There are no significant differences in any of these categories

### Checking if there was a difference in Mental Modals

In [34]:
def collect_mms(surveys, black_list):
    mental_models = {
        "natural language": [],
        "keywords": [],
        "specific question": [],
        "follow-up questions": [],
        "general answer": [],
        "personalized answer": [],
        "immediate answer": [],
        "long dialog": []
    }

    labels_map  = {'chat_exp_1': 'natural language',
                   'chat_exp_2': 'keywords',
                   'chat_exp_3': 'specific question',
                   'chat_exp_4': 'follow-up questions',
                   'chat_exp_5': 'general answer',
                   'chat_exp_6': 'personalized answer',
                   'chat_exp_7': 'immediate answer',
                   'chat_exp_8': 'long dialog'}
    user_mms = {}


    for res in surveys:
        user = res["user"]
        if user not in black_list:
            for key in labels_map:
                if not res[key] == 'None':
                    mental_models[labels_map[key]].append(int(res[key]))
            user_mms[user] = {key: mental_models[key][-1] for key in mental_models}
        else:
            print(user)
    return mental_models, user_mms

In [35]:
base_mental_models, base_user_mms = collect_mms(baseline_post_surveys, base_blacklist)
exp_mental_models, exp_user_mms = collect_mms(exp_post_surveys, exp_blacklist)

1fd4736bcd456d02458c2a32944e60


In [37]:
for key in base_mental_models:
    print(key)
    print(f"BASELINE: {sum(base_mental_models[key])/len(base_mental_models[key])}, EXP: {sum(exp_mental_models[key])/len(exp_mental_models[key])}")
    print(mannwhitneyu(base_mental_models[key], exp_mental_models[key]))

natural language
BASELINE: 3.0317460317460316, EXP: 3.292307692307692
MannwhitneyuResult(statistic=1798.5, pvalue=0.22206975409305985)
keywords
BASELINE: 3.765625, EXP: 3.515625
MannwhitneyuResult(statistic=2204.5, pvalue=0.438963749720486)
specific question
BASELINE: 3.625, EXP: 3.859375
MannwhitneyuResult(statistic=1768.5, pvalue=0.1659022953505358)
follow-up questions
BASELINE: 2.9836065573770494, EXP: 3.0806451612903225
MannwhitneyuResult(statistic=1805.5, pvalue=0.6591249805802095)
general answer
BASELINE: 3.8412698412698414, EXP: 3.725806451612903
MannwhitneyuResult(statistic=2004.5, pvalue=0.7925506202903663)
personalized answer
BASELINE: 2.375, EXP: 3.015873015873016
MannwhitneyuResult(statistic=1524.0, pvalue=0.015019283015731263)
immediate answer
BASELINE: 3.5555555555555554, EXP: 3.8461538461538463
MannwhitneyuResult(statistic=1701.0, pvalue=0.08615834990266606)
long dialog
BASELINE: 3.180327868852459, EXP: 2.875
MannwhitneyuResult(statistic=2200.5, pvalue=0.2109843382765673

In [39]:
def collect_mms_by_condition(surveys, black_list):
    mental_models = {
        "cts":
            {
            "natural language": [],
            "keywords": [],
            "specific question": [],
            "follow-up questions": [],
            "general answer": [],
            "personalized answer": [],
            "immediate answer": [],
            "long dialog": []
        },
        "faq":
            {
            "natural language": [],
            "keywords": [],
            "specific question": [],
            "follow-up questions": [],
            "general answer": [],
            "personalized answer": [],
            "immediate answer": [],
            "long dialog": []
        },
        "hdc":
            {
            "natural language": [],
            "keywords": [],
            "specific question": [],
            "follow-up questions": [],
            "general answer": [],
            "personalized answer": [],
            "immediate answer": [],
            "long dialog": []
        }
    }


    labels_map  = {'chat_exp_1': 'natural language',
                   'chat_exp_2': 'keywords',
                   'chat_exp_3': 'specific question',
                   'chat_exp_4': 'follow-up questions',
                   'chat_exp_5': 'general answer',
                   'chat_exp_6': 'personalized answer',
                   'chat_exp_7': 'immediate answer',
                   'chat_exp_8': 'long dialog'}

    for res in surveys:
        user = res["user"]
        condition = user_condition_mapping[user]
        if user not in black_list:
            for key in labels_map:
                if not res[key] == "None":
                    mental_models[condition][labels_map[key]].append(int(res[key]))
        else:
            print(user)
    return mental_models

In [40]:
base_mental_models = collect_mms_by_condition(baseline_post_surveys, base_blacklist)
exp_mental_models = collect_mms_by_condition(exp_post_surveys, exp_blacklist)

1fd4736bcd456d02458c2a32944e60


In [41]:
for condition in base_mental_models:
    print(condition)
    for key in base_mental_models[condition]:
        print(key)
        print(f"BASELINE: {sum(base_mental_models[condition][key])/len(base_mental_models[condition][key])}, EXP: {sum(exp_mental_models[condition][key])/len(exp_mental_models[condition][key])}")
        print(ttest_ind(base_mental_models[condition][key], exp_mental_models[condition][key]))

cts
natural language
BASELINE: 3.238095238095238, EXP: 3.590909090909091
Ttest_indResult(statistic=-1.0786639605735173, pvalue=0.28704282396421676)
keywords
BASELINE: 3.7142857142857144, EXP: 3.590909090909091
Ttest_indResult(statistic=0.32469059477532397, pvalue=0.7470661698965606)
specific question
BASELINE: 3.4761904761904763, EXP: 3.909090909090909
Ttest_indResult(statistic=-1.2972372637042102, pvalue=0.201801939748726)
follow-up questions
BASELINE: 3.8095238095238093, EXP: 3.9047619047619047
Ttest_indResult(statistic=-0.2630668208823285, pvalue=0.7938494301451543)
general answer
BASELINE: 3.619047619047619, EXP: 3.4761904761904763
Ttest_indResult(statistic=0.35985608634244015, pvalue=0.7208478340117593)
personalized answer
BASELINE: 3.0, EXP: 3.5238095238095237
Ttest_indResult(statistic=-1.3499219076029938, pvalue=0.18463104216642492)
immediate answer
BASELINE: 3.619047619047619, EXP: 4.090909090909091
Ttest_indResult(statistic=-1.7300205288018635, pvalue=0.09114887172600546)
long

### What role do expectations have on success?

In [91]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user in user_black_list:
            continue
        success = 1 if d["end_condition"] == "SUCCESS" else 0
        if user_mms[user][key] >= 3:
            yes.append(success)
        else:
            no.append(success)
    res = barnard_exact([yes.count(1), yes.count(0)], [no.count(1), no.count(0)])
    print(f" {key}: {res}")

 natural language: Ttest_indResult(statistic=-1.2615625894698697, pvalue=0.2116850113328921)
 keywords: Ttest_indResult(statistic=0.6817786549807436, pvalue=0.4978388353437795)
 specific question: Ttest_indResult(statistic=0.8100219620698343, pvalue=0.4209292172076745)
 follow-up questions: Ttest_indResult(statistic=2.1792641132530566, pvalue=0.03299950024309895)
 general answer: Ttest_indResult(statistic=-0.5999062719669357, pvalue=0.5506869025902197)
 personalized answer: Ttest_indResult(statistic=-1.0559774152551655, pvalue=0.29494847330721174)
 immediate answer: Ttest_indResult(statistic=1.5375601245080888, pvalue=0.12908585919722063)
 long dialog: Ttest_indResult(statistic=-1.104090331129679, pvalue=0.2736878615942981)


Mental models have no impact on actual success in the cts setting

### Role of Mental Models on Usability

In [81]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_usability:
            continue
        usability = u_usability[user]
        if user_mms[user][key] >= 3:
            yes.append(usability)
        else:
            no.append(usability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=-1.4321320775297293, pvalue=0.15738289394837543)
55.96590909090909 64.70588235294117
 keywords: Ttest_indResult(statistic=-1.090277005916156, pvalue=0.28002403112957736)
56.53409090909091 63.23529411764706
 specific question: Ttest_indResult(statistic=0.3830301576588764, pvalue=0.7030739866630724)
58.92857142857143 56.25
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
58.40163934426229 nan
 general answer: Ttest_indResult(statistic=-0.6474291339110548, pvalue=0.5198655421789713)
56.8014705882353 60.416666666666664
 personalized answer: Ttest_indResult(statistic=-0.39706337358767435, pvalue=0.6927534714196355)
57.03125 59.29054054054054
 immediate answer: Ttest_indResult(statistic=-2.621444821675263, pvalue=0.011120154098656935)
55.52884615384615 75.0
 long dialog: Ttest_indResult(statistic=3.5493223590857466, pvalue=0.0007654443048735936)
61.36363636363637 31.25


Natural Language/Keyword expectation and dialog length expectation had a significant effect on usability

### Role of Mental Models on Reliability

In [82]:
for key in mental_models:
    if key == "specific question":
        continue
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_reliability:
            continue
        reliability = u_reliability[user]
        if user_mms[user][key] >= 3:
            yes.append(reliability)
        else:
            no.append(reliability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=-1.0304388908112347, pvalue=0.3070097939761558)
2.7462121212121215 2.9509803921568625
 keywords: Ttest_indResult(statistic=-0.8916583847186615, pvalue=0.37619815459541883)
2.753787878787879 2.9313725490196076
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
2.803278688524589 nan
 general answer: Ttest_indResult(statistic=0.06921970297373171, pvalue=0.9450488122042161)
2.808823529411765 2.796296296296296
 personalized answer: Ttest_indResult(statistic=1.0248705257172168, pvalue=0.30960803740759535)
2.9166666666666665 2.7297297297297303
 immediate answer: Ttest_indResult(statistic=-0.13911303633432043, pvalue=0.889834594357785)
2.798076923076922 2.8333333333333335
 long dialog: Ttest_indResult(statistic=2.8175491426145993, pvalue=0.006573857524343258)
2.8818181818181814 2.083333333333333


Natural language and dialog length had a significant effect on pereceived reliability

### Effect of mental models on trust

In [83]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_trust:
            continue
        trust = u_trust[user]
        if user_mms[user][key] >= 3:
            yes.append(trust)
        else:
            no.append(trust)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=0.8603481533906172, pvalue=0.3930798471930703)
2.9204545454545454 2.7058823529411766
 keywords: Ttest_indResult(statistic=-0.11987462658573536, pvalue=0.9049897340241074)
2.852272727272727 2.8823529411764706
 specific question: Ttest_indResult(statistic=1.621143018968216, pvalue=0.11031970914810925)
2.9489795918367347 2.5
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
2.860655737704918 nan
 general answer: Ttest_indResult(statistic=0.6587876603499426, pvalue=0.5125950253555297)
2.926470588235294 2.7777777777777777
 personalized answer: Ttest_indResult(statistic=0.5514486068116325, pvalue=0.5834087447901244)
2.9375 2.810810810810811
 immediate answer: Ttest_indResult(statistic=0.7203955887623041, pvalue=0.4741260286321175)
2.894230769230769 2.6666666666666665
 long dialog: Ttest_indResult(statistic=2.6752061439937114, pvalue=0.009649915824434006)
2.9545454545454546 2.0


Natural langauge and expected dialog length had a significant effect on trust

In [54]:
post_mental_models = {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
}

post_user_mms = {}

for res in post_surveys:
    user = res["user"]
    if user in user_black_list:
            continue
    if res["chat_exp_1"] != "None":
        post_mental_models["natural language"].append(int(res["chat_exp_1"]))
    if res["chat_exp_2"] != "None":
        post_mental_models["keywords"].append(int(res["chat_exp_2"]))
    if res["chat_exp_3"] != "None":
        post_mental_models["specific question"].append(int(res["chat_exp_3"]))
    if res["chat_exp_4"] != "None":
        post_mental_models["follow-up questions"].append(int(res['chat_exp_4']))
    if res["chat_exp_5"] != "None":
        post_mental_models["general answer"].append(int(res["chat_exp_5"]))
    if res["chat_exp_6"] != "None":
        post_mental_models["personalized answer"].append(int(res["chat_exp_6"]))
    if res["chat_exp_7"] != "None":
        post_mental_models["immediate answer"].append(int(res['chat_exp_7']))
    if res["chat_exp_8"] != "None":
        post_mental_models["long dialog"].append(int(res["chat_exp_8"]))
    post_user_mms[user] = {key: post_mental_models[key][-1] for key in post_mental_models}
    
labels = [key for key in post_mental_models]
avg_post_mental_models = [np.mean(post_mental_models[l]) for l in labels]

In [55]:
for key in mental_models:
    print({f"{key}: {stats.ttest_ind(mental_models[key], post_mental_models[key])}"})

{'natural language: Ttest_indResult(statistic=0.28033098596050277, pvalue=0.7806683492514809)'}
{'keywords: Ttest_indResult(statistic=-0.9543482955111211, pvalue=0.34563950251914166)'}
{'specific question: Ttest_indResult(statistic=2.4237726026264967, pvalue=0.01997534226520323)'}
{'follow-up questions: Ttest_indResult(statistic=-1.294369603381147, pvalue=0.20296075493839177)'}
{'general answer: Ttest_indResult(statistic=-0.6412364700532214, pvalue=0.5250264189676689)'}
{'personalized answer: Ttest_indResult(statistic=-1.4509525002200236, pvalue=0.15459078143343824)'}
{'immediate answer: Ttest_indResult(statistic=0.4965635331614206, pvalue=0.6222149306594651)'}
{'long dialog: Ttest_indResult(statistic=-1.3286579139913106, pvalue=0.1916830191682154)'}


People thought that they needed to ask more general questions after interacting with the chatbot, but otherwise there were no significant changes in mental models